# Lyα forest tomography from random transverse sightlines

This notebook shoots randomly placed, **z-directed** sightlines through a held-out CAMELS volume. Their transverse sampling has a nominal spacing of 2 h^-1 cMpc. Each skewer is inverted with direct FGPA, regularized MAP, and a compact neural network; the sparse one-dimensional reconstructions are then converted into 3D maps with a Wiener posterior mean.

The forward mock uses the co-spatial CAMELS neutral-hydrogen density and temperature fields in a continuous Voigt calculation. The traditional inversions intentionally do not receive either hidden field.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from scipy import ndimage, special
from scipy.optimize import brentq, least_squares
from astropy import units as u
from astropy.constants import c, k_B, m_e, m_p
from astropy.cosmology import Planck18 as cosmo
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

rng = np.random.default_rng(2026)
torch.manual_seed(2026)
torch.set_num_threads(min(4, torch.get_num_threads()))
device = torch.device('cpu')

notebook_directory = Path('.') if Path('Sims').exists() else Path('Hands-On')
data_directory = notebook_directory / 'Sims/CMD_z=2_grid128'
output_directory = notebook_directory / 'figures'
output_directory.mkdir(exist_ok=True)

box_size, redshift = 25.0, 2.0
test_simulation = 0
target_transverse_spacing = 2.0  # h^-1 cMpc
signal_to_noise, instrument_fwhm_kms = 5.0, 120.0
inversion_T0_K, inversion_gamma = 10_000.0, 1.5
neural_train_per_box, neural_epochs = 32, 25
wiener_transverse_scale, wiener_los_scale = 2.0, 1.0  # h^-1 cMpc
wiener_noise_std = 0.45


## 1. Load the held-out volume and select random $z$-directed sightlines

The sampler draws unique transverse grid cells uniformly at random. Its tomography sampling scale is defined conventionally as the square root of area per sightline, sqrt(A/N), and is set to 2 h^-1 cMpc.

In [ ]:
gas_boxes = np.load(data_directory / 'Grids_Mgas_IllustrisTNG_CV_128_z=2.0.npy', mmap_mode='r')
dm_boxes = np.load(data_directory / 'Grids_Mcdm_IllustrisTNG_CV_128_z=2.0.npy', mmap_mode='r')
HI_boxes = np.load(data_directory / 'Grids_HI_IllustrisTNG_CV_128_z=2.0.npy', mmap_mode='r')
temperature_boxes = np.load(data_directory / 'Grids_T_IllustrisTNG_CV_128_z=2.0.npy', mmap_mode='r')

N = gas_boxes.shape[1]
dx = box_size / N
coordinates = (np.arange(N) + 0.5) * dx
gas_means = np.array([box.mean(dtype=np.float64) for box in gas_boxes])
dm_means = np.array([box.mean(dtype=np.float64) for box in dm_boxes])
Delta_dm_true = dm_boxes[test_simulation].astype(np.float64) / dm_means[test_simulation]

def periodic_distance(points_a, points_b):
    delta = np.abs(points_a[:, None, :] - points_b[None, :, :])
    delta = np.minimum(delta, box_size - delta)
    return np.sqrt(np.sum(delta**2, axis=-1))

def random_sightlines(target_spacing):
    n_lines = round((box_size / target_spacing) ** 2)
    flat_indices = rng.choice(N * N, size=n_lines, replace=False)
    return np.column_stack(np.divmod(flat_indices, N))

sightline_indices = random_sightlines(target_transverse_spacing)
sightline_xy = (sightline_indices + 0.5) * dx
separations = periodic_distance(sightline_xy, sightline_xy)
np.fill_diagonal(separations, np.inf)
mean_nearest_separation = separations.min(axis=1).mean()
mean_transverse_spacing = np.sqrt(box_size**2 / len(sightline_indices))
print(f'{len(sightline_indices)} uniformly random z-directed sightlines; sqrt(A/N) = {mean_transverse_spacing:.2f} h^-1 cMpc')
print(f'Mean nearest-neighbour separation (diagnostic) = {mean_nearest_separation:.2f} h^-1 cMpc')


### Sampling diagnostic

The first checkpoint shows where the random sightlines pierce a representative transverse slice. Clustering and gaps are expected for uniform random sampling.

In [ ]:
z_slice = N // 2
fig, axis = plt.subplots(figsize=(6.5, 5.5), constrained_layout=True)
image = axis.imshow(
    np.log10(np.clip(Delta_dm_true[:, :, z_slice], 1e-3, None)).T,
    origin='lower', extent=(0, box_size, 0, box_size), cmap='magma'
)
axis.scatter(sightline_xy[:, 0], sightline_xy[:, 1], s=18, facecolors='none', edgecolors='cyan', linewidths=.8)
axis.set(xlabel='x [h$^{-1}$ cMpc]', ylabel='y [h$^{-1}$ cMpc]', title=f'True DM slice and {len(sightline_xy)} sightlines')
fig.colorbar(image, ax=axis, label=r'$\log_{10}\Delta_{\rm dm}$')
plt.show()


## 2. Physical HI-and-temperature mock spectra

Every absorber cell contributes a thermally broadened Voigt profile. The function is batched over sightlines so the neural training data and the tomography mock use the same forward physics.

In [ ]:
hydrogen_mass_g = (m_p + m_e).to_value(u.g)
density_unit = (u.Msun / cosmo.h) / (u.Mpc / cosmo.h) ** 3
H_z = cosmo.H(redshift).to_value(u.km / u.s / u.Mpc)
z_abs = redshift + H_z * ((coordinates - box_size / 2) / cosmo.h) / c.to_value(u.km / u.s)
velocity_pixel_width = H_z * (dx / cosmo.h) / (1 + redshift)
sigma_pixels = instrument_fwhm_kms / (2 * np.sqrt(2 * np.log(2))) / velocity_pixel_width
K = ndimage.gaussian_filter1d(np.eye(N), sigma_pixels, axis=0, mode='reflect')
noise_sigma = 1 / signal_to_noise

def number_density_HI(rho_HI):
    comoving = (rho_HI * density_unit).to_value(u.g / u.cm**3)
    return comoving * (1 + z_abs) ** 3 / hydrogen_mass_g

def voigt_tau_batch(n_HI, temperature):
    c_cms, I_alpha, gamma_alpha = c.to_value(u.cm / u.s), 4.45e-18, 6.262e8
    nu_alpha = c_cms / (1215.67e-8)
    b = np.sqrt(2 * k_B.to_value(u.erg / u.K) * temperature / hydrogen_mass_g)
    damping = gamma_alpha * c_cms / (4 * np.pi * nu_alpha * b)
    velocity = c_cms * (z_abs[:, None] - z_abs[None, :]) / (1 + z_abs[None, :])
    profile = np.real(special.wofz(velocity[None] / b[:, :, None] + 1j * damping[:, :, None]))
    width_cm = dx * u.Mpc.to(u.cm) / cosmo.h
    prefactor = c_cms * I_alpha * width_cm * n_HI / (np.sqrt(np.pi) * b * (1 + z_abs))
    return np.sum(prefactor[:, :, None] * profile, axis=1)

def observe_HI_temperature(rho_HI, temperature, add_noise=True, batch_size=64):
    flux = np.empty(rho_HI.shape, dtype=np.float64)
    for start in range(0, len(rho_HI), batch_size):
        stop = min(start + batch_size, len(rho_HI))
        tau = voigt_tau_batch(number_density_HI(rho_HI[start:stop]), temperature[start:stop])
        flux[start:stop] = np.exp(-tau) @ K.T
    if add_noise:
        flux += rng.normal(0, noise_sigma, flux.shape)
    return flux

ix, iy = sightline_indices.T
rho_HI_lines = HI_boxes[test_simulation, ix, iy, :].astype(np.float64)
temperature_lines = temperature_boxes[test_simulation, ix, iy, :].astype(np.float64)
Delta_b_lines = gas_boxes[test_simulation, ix, iy, :].astype(np.float64) / gas_means[test_simulation]
Delta_dm_lines = dm_boxes[test_simulation, ix, iy, :].astype(np.float64) / dm_means[test_simulation]
flux_lines = observe_HI_temperature(rho_HI_lines, temperature_lines)


### Mock-spectrum checkpoint

These examples make the noise level, instrumental smoothing, and relation between absorption and the underlying density visible before any inversion is attempted.

In [ ]:
example_lines = np.linspace(0, len(sightline_indices) - 1, 3, dtype=int)
fig, axes = plt.subplots(len(example_lines), 1, figsize=(10, 7), sharex=True, constrained_layout=True)
for axis, line_index in zip(axes, example_lines):
    axis.plot(coordinates, flux_lines[line_index], color='tab:blue', lw=1, label='Noisy flux')
    axis.set_ylim(-0.15, 1.25)
    density_axis = axis.twinx()
    density_axis.plot(coordinates, Delta_dm_lines[line_index], color='tab:orange', lw=1, alpha=.7, label=r'True $\Delta_{\rm dm}$')
    density_axis.set_yscale('log')
    density_axis.set_ylim(.05, 30)
    axis.set_ylabel('Flux')
    density_axis.set_ylabel(r'$\Delta_{\rm dm}$')
    axis.set_title(f'Sightline {line_index}: (x, y) = ({sightline_xy[line_index, 0]:.1f}, {sightline_xy[line_index, 1]:.1f})')
axes[0].legend(loc='lower left')
axes[-1].set_xlabel('z [h$^{-1}$ cMpc]')
plt.show()


## 3. Direct FGPA and regularized-MAP skewer inversions

The FGPA amplitude is calibrated on physical mocks from the 26 non-test boxes. Both traditional methods return a baryons-trace-DM proxy. MAP additionally fits the resolution operator and a curvature prior in log density.

In [ ]:
training_simulations = np.arange(1, gas_boxes.shape[0])
temperature_reference = 10_000.0
beta = 2 - 0.7 * (inversion_gamma - 1)
thermal_normalization = (inversion_T0_K / temperature_reference) ** -0.7
calibration_density = np.concatenate([gas_boxes[s, :, 64, 64].astype(np.float64) / gas_means[s] for s in training_simulations])
calibration_temperature = np.concatenate([temperature_boxes[s, :, 64, 64].astype(np.float64) for s in training_simulations])
calibration_shape = calibration_density**2 * (calibration_temperature / temperature_reference) ** -0.7
calibration_flux = observe_HI_temperature(HI_boxes[training_simulations, :, 64, 64].astype(np.float64), temperature_boxes[training_simulations, :, 64, 64].astype(np.float64), add_noise=False)
mean_physical_flux = calibration_flux.mean()
A_fgpa = brentq(lambda A: np.exp(-A * calibration_shape).mean() - mean_physical_flux, 1e-6, 100)

L = np.zeros((N - 2, N))
for row in range(N - 2):
    L[row, row:row+3] = (1, -2, 1)

def direct_fgpa(flux):
    clipped = np.clip(flux, noise_sigma, 1 - 1e-8)
    return (-np.log(clipped) / (A_fgpa * thermal_normalization)) ** (1 / beta)

def map_fgpa(flux, regularization=1.0):
    direct = direct_fgpa(flux)
    def model_and_jacobian(log_density):
        tau = A_fgpa * thermal_normalization * np.exp(beta * log_density)
        intrinsic = np.exp(-tau)
        derivative = -beta * tau * intrinsic
        return K @ intrinsic, K * derivative[None, :]
    def residual(log_density):
        model, _ = model_and_jacobian(log_density)
        return np.concatenate(((model - flux) / noise_sigma, np.sqrt(regularization) * (L @ log_density)))
    def jacobian(log_density):
        _, jac = model_and_jacobian(log_density)
        return np.vstack((jac / noise_sigma, np.sqrt(regularization) * L))
    result = least_squares(residual, np.log(np.clip(ndimage.gaussian_filter1d(direct, 1), .03, 20)), jac=jacobian, bounds=(np.log(.03), np.log(20)), max_nfev=60)
    return np.exp(result.x)

Delta_dm_direct_lines = np.asarray([direct_fgpa(flux) for flux in flux_lines])
Delta_dm_map_lines = np.asarray([map_fgpa(flux) for flux in flux_lines])
print(f'A_FGPA={A_fgpa:.4f}; physical calibration mean flux={mean_physical_flux:.3f}')


### Traditional inversion checkpoint

Direct FGPA responds pixel by pixel, whereas the MAP estimate explicitly accounts for spectral resolution and suppresses unstable curvature.

In [ ]:
fig, axes = plt.subplots(len(example_lines), 1, figsize=(10, 7), sharex=True, sharey=True, constrained_layout=True)
for axis, line_index in zip(axes, example_lines):
    axis.plot(coordinates, Delta_dm_lines[line_index], color='black', lw=1.6, label='Truth')
    axis.plot(coordinates, Delta_dm_direct_lines[line_index], lw=1, alpha=.8, label='Direct FGPA')
    axis.plot(coordinates, Delta_dm_map_lines[line_index], lw=1.2, label='Regularized MAP')
    axis.set_yscale('log')
    axis.set_ylim(.03, 30)
    axis.set_ylabel(r'$\Delta_{\rm dm}$')
axes[0].legend(ncol=3, fontsize=8)
axes[-1].set_xlabel('z [h$^{-1}$ cMpc]')
plt.show()


## 4. A compact neural DM inversion

The network directly maps a noisy physical flux skewer to log Delta_dm. Training samples come only from boxes 1–26, so box 0 and all tomography sightlines remain held out.

In [ ]:
def draw_training_skewers():
    densities, HI, temperatures = [], [], []
    for simulation in training_simulations:
        flats = rng.choice(N*N, neural_train_per_box, replace=False)
        x_train, y_train = np.divmod(flats, N)
        densities.append(dm_boxes[simulation, x_train, y_train, :].astype(np.float64) / dm_means[simulation])
        HI.append(HI_boxes[simulation, x_train, y_train, :].astype(np.float64))
        temperatures.append(temperature_boxes[simulation, x_train, y_train, :].astype(np.float64))
    return np.concatenate(densities), np.concatenate(HI), np.concatenate(temperatures)

train_dm, train_HI, train_temperature = draw_training_skewers()
train_flux = observe_HI_temperature(train_HI, train_temperature)
flux_mean, flux_std = train_flux.mean(), train_flux.std()
target = np.log(np.clip(train_dm, .03, 20))
target_mean, target_std = target.mean(), target.std()

class NeuralDMInverter(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.Sequential(nn.Conv1d(1, 24, 7, padding=3), nn.GELU(), nn.Conv1d(24, 24, 7, padding=3), nn.GELU(), nn.Conv1d(24, 1, 1))
    def forward(self, flux):
        return self.layers(flux)

network = NeuralDMInverter().to(device)
dataset = TensorDataset(torch.tensor((train_flux-flux_mean)/flux_std, dtype=torch.float32)[:, None], torch.tensor((target-target_mean)/target_std, dtype=torch.float32)[:, None])
loader = DataLoader(dataset, batch_size=64, shuffle=True)
optimizer = torch.optim.AdamW(network.parameters(), lr=2e-3, weight_decay=1e-4)
training_loss = []
for epoch in range(neural_epochs):
    epoch_loss = []
    for flux_batch, target_batch in loader:
        optimizer.zero_grad()
        loss = torch.mean((network(flux_batch) - target_batch)**2)
        loss.backward()
        optimizer.step()
        epoch_loss.append(loss.item())
    training_loss.append(np.mean(epoch_loss))

with torch.no_grad():
    neural_log_dm = network(torch.tensor((flux_lines-flux_mean)/flux_std, dtype=torch.float32)[:, None]).squeeze(1).numpy() * target_std + target_mean
Delta_dm_neural_lines = np.exp(neural_log_dm)
print(f'Neural training skewers: {len(train_flux)}; final epoch MSE={training_loss[-1]:.4f}')


### Neural-inversion checkpoint

The loss curve checks optimization, while the held-out skewers show whether the network generalizes beyond its training simulations.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2), constrained_layout=True)
axes[0].plot(np.arange(1, neural_epochs + 1), training_loss, marker='o', ms=3)
axes[0].set(xlabel='Epoch', ylabel='Mean training MSE', title='Neural optimization')
axes[0].set_yscale('log')
line_index = example_lines[1]
axes[1].plot(coordinates, Delta_dm_lines[line_index], color='black', lw=1.6, label='Truth')
axes[1].plot(coordinates, Delta_dm_direct_lines[line_index], alpha=.7, label='Direct FGPA')
axes[1].plot(coordinates, Delta_dm_map_lines[line_index], label='MAP')
axes[1].plot(coordinates, Delta_dm_neural_lines[line_index], label='Neural')
axes[1].set(xlabel='z [h$^{-1}$ cMpc]', ylabel=r'$\Delta_{\rm dm}$', title='Held-out sightline comparison')
axes[1].set_yscale('log')
axes[1].set_ylim(.03, 30)
axes[1].legend(fontsize=8)
for axis in axes:
    axis.grid(alpha=.2)
plt.show()


## 5. Three-dimensional Wiener tomography

The neural skewer estimates are converted into a 3D map with a Wiener posterior mean. The prior covariance is separable in transverse and line-of-sight directions, so the calculation uses two small eigendecompositions rather than a dense 20,000 by 20,000 solve. The reconstructed map is not Gaussian-smoothed after tomography.

In [ ]:
def periodic_1d_distance(a, b):
    distance = np.abs(a[:, None] - b[None, :])
    return np.minimum(distance, box_size - distance)

def wiener_tomography(line_delta):
    variance = np.var(Delta_dm_true - 1)
    C_lines = np.exp(-0.5 * (periodic_distance(sightline_xy, sightline_xy) / wiener_transverse_scale)**2)
    C_z = variance * np.exp(-0.5 * (periodic_1d_distance(coordinates, coordinates) / wiener_los_scale)**2)
    lambda_xy, U_xy = np.linalg.eigh(C_lines)
    lambda_z, U_z = np.linalg.eigh(C_z)
    data = line_delta - 1
    transformed = U_xy.T @ data @ U_z
    coefficients = transformed / (lambda_xy[:, None] * lambda_z[None, :] + wiener_noise_std**2)
    grid_xy = np.stack(np.meshgrid(coordinates, coordinates, indexing='ij'), axis=-1).reshape(-1, 2)
    C_grid_lines = np.exp(-0.5 * (periodic_distance(grid_xy, sightline_xy) / wiener_transverse_scale)**2)
    posterior = C_grid_lines @ (U_xy @ coefficients @ U_z.T) @ C_z
    return 1 + posterior.reshape(N, N, N)

dm_neural_3d = wiener_tomography(Delta_dm_neural_lines)  # no post-reconstruction smoothing


## 6. Visual and summary-statistic comparisons

The native CAMELS truth and unsmoothed neural reconstruction are each shown alongside a periodic Gaussian-smoothed copy with sigma = 2 h^-1 cMpc. The original reconstructed volume remains unchanged; smoothing is applied only to a separate comparison copy. All spectra use five logarithmic bins from k=0.5 to 5 h Mpc^-1. The bispectrum diagnostic is a matched equilateral-shell estimator, so its relative comparison is meaningful even though it is not a full triangle-bin survey.

In [ ]:
truth_smoothing_scale = 2.0  # Gaussian sigma in h^-1 cMpc
Delta_dm_true_smoothed = ndimage.gaussian_filter(
    Delta_dm_true, sigma=truth_smoothing_scale / dx, mode='wrap'
)
dm_neural_3d_smoothed = ndimage.gaussian_filter(
    dm_neural_3d, sigma=truth_smoothing_scale / dx, mode='wrap'
)
maps = [
    ('True DM (native resolution)', Delta_dm_true),
    (r'True DM ($\sigma=2\ h^{-1}\,\mathrm{cMpc}$)', Delta_dm_true_smoothed),
    ('Neural tomography (unsmoothed)', dm_neural_3d),
    (r'Neural tomography ($\sigma=2\ h^{-1}\,\mathrm{cMpc}$)', dm_neural_3d_smoothed),
]
fig, axes = plt.subplots(2, 2, figsize=(12, 11), sharex=True, sharey=True, constrained_layout=True)
limits = np.quantile(np.log10(np.clip(Delta_dm_true[:, :, z_slice], 1e-3, None)), (.02, .98))
for axis, (name, volume) in zip(axes.flat, maps):
    image = axis.imshow(np.log10(np.clip(volume[:, :, z_slice], 1e-3, None)).T, origin='lower', extent=(0, box_size, 0, box_size), vmin=limits[0], vmax=limits[1], cmap='magma')
    axis.set(title=name, xlabel='x [h^-1 cMpc]', ylabel='y [h^-1 cMpc]')
for axis in axes[1]:
    axis.scatter(sightline_xy[:, 0], sightline_xy[:, 1], s=9, facecolors='none', edgecolors='cyan', linewidths=.55, label='z sightlines')
axes[1, 0].legend(loc='upper right', fontsize=8)
fig.colorbar(image, ax=axes, label='log10 Delta_dm')
fig.savefig(output_directory / 'tomography_slices_and_sightlines.png', dpi=150, bbox_inches='tight')
plt.show()

def binned_power_3d(volume, n_bins=5, k_min=.5, k_max=5.):
    contrast = volume / volume.mean() - 1
    modes = 2*np.pi*np.fft.fftfreq(N, d=dx)
    kx, ky, kz = np.meshgrid(modes, modes, modes, indexing='ij')
    k = np.sqrt(kx**2 + ky**2 + kz**2)
    power = dx**3 / N**3 * np.abs(np.fft.fftn(contrast))**2
    edges = np.logspace(np.log10(k_min), np.log10(k_max), n_bins+1)
    centres = np.sqrt(edges[:-1] * edges[1:])
    return centres, np.array([power[(k >= lo) & (k < hi)].mean() for lo, hi in zip(edges[:-1], edges[1:])]), edges, k

def equilateral_shell_bispectrum(volume, edges, k):
    contrast = volume / volume.mean() - 1
    transformed = np.fft.fftn(contrast)
    return np.array([box_size**3 * np.mean(np.fft.ifftn(transformed * ((k >= lo) & (k < hi))).real**3) for lo, hi in zip(edges[:-1], edges[1:])])

k_centres, power_true, k_edges, k_magnitude = binned_power_3d(Delta_dm_true)
truth_label = maps[0][0]
power_results = {truth_label: power_true}
bispectrum_results = {truth_label: equilateral_shell_bispectrum(Delta_dm_true, k_edges, k_magnitude)}
for name, volume in maps[1:]:
    power_results[name] = binned_power_3d(volume)[1]
    bispectrum_results[name] = equilateral_shell_bispectrum(volume, k_edges, k_magnitude)

fig, axes = plt.subplots(1, 3, figsize=(17, 4.6), constrained_layout=True)
bins = np.linspace(-2.2, 1.3, 35)
for name, volume in maps:
    axes[0].hist(np.log10(np.clip(volume, 1e-3, None)).ravel(), bins=bins, density=True, histtype='step', lw=1.7, label=name)
    axes[1].loglog(k_centres, power_results[name], marker='o', label=name)
    axes[2].semilogy(k_centres, np.abs(bispectrum_results[name]), marker='o', label=name)
axes[0].set(xlabel='log10 Delta_dm', ylabel='PDF', title='One-point PDF')
axes[1].set(xlabel='k [h Mpc^-1]', ylabel='P(k)', title='3D power spectrum')
axes[2].set(xlabel='k [h Mpc^-1]', ylabel='|B_eq(k)|', title='Equilateral-shell bispectrum')
for axis in axes:
    axis.grid(alpha=.2)
    axis.legend(fontsize=7)
fig.savefig(output_directory / 'tomography_pdf_power_bispectrum.png', dpi=150, bbox_inches='tight')
plt.show()
